# День 2. Векторные представления текста и базовые NLP-модели

Практический блокнот: сходство текстов, семантические признаки и простая классификация. Примеры ориентированы на учебные задачи, в том числе финансовые тексты.

**Цели:** сравнить документы по TF–IDF, понять идею эмбеддингов, построить базовый классификатор и обсудить ограничения метрик.

## 1. Учебный корпус с метками

Метки условные: `risk` — риск/кредитование, `service` — обслуживание клиентов. Корпус намеренно мал и предназначен только для демонстрации процесса.

In [ ]:
texts = [
    "Банк оценивает кредитный риск заёмщика",
    "Модель прогнозирует вероятность дефолта клиента",
    "Аналитик изучает просроченную задолженность",
    "Клиент открывает счёт в мобильном приложении",
    "Банк улучшает качество обслуживания клиентов",
    "Мобильный сервис помогает управлять картой",
    "Кредитный скоринг оценивает риск невозврата",
    "Поддержка отвечает на вопросы пользователей"
]
labels = ["risk", "risk", "risk", "service", "service", "service", "risk", "service"]
list(zip(texts, labels))

## 2. TF–IDF и косинусное сходство

Косинусное сходство сравнивает направление двух векторов. Для TF–IDF-векторов оно помогает находить документы с похожим набором значимых слов. Это не гарантирует смысловую эквивалентность: синонимы могут не совпасть по словам.

In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b", ngram_range=(1, 2))
    X = vectorizer.fit_transform(texts)
    query = "оценка кредитного риска клиента"
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, X).ravel()
    for idx in scores.argsort()[::-1][:4]:
        print(f"{scores[idx]:.3f} | {labels[idx]:7s} | {texts[idx]}")
except ImportError:
    print("Установите scikit-learn: python -m pip install scikit-learn")

### Задание на бумаге
Для векторов A=(1, 1, 0) и B=(1, 0, 1) вычислите косинусное сходство по формуле
\(\cos(A,B)=\frac{A\cdot B}{\|A\|\|B\|}\).
Что изменится, если B=(2, 2, 0)? Почему косинус не зависит от масштаба вектора?

## 3. Классификация текста

Разделим данные на обучающую и тестовую части и обучим логистическую регрессию. На маленьком корпусе оценка качества нестабильна; в реальном проекте нужны больше данных, корректная схема валидации и анализ ошибок.

In [ ]:
try:
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import make_pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, accuracy_score

    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=0.25, random_state=42, stratify=labels
    )
    model = make_pipeline(
        TfidfVectorizer(token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b"),
        LogisticRegression(max_iter=1000)
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print("Accuracy:", round(accuracy_score(y_test, pred), 3))
    print(classification_report(y_test, pred, zero_division=0))
except ImportError:
    print("Установите scikit-learn: python -m pip install scikit-learn")

## 4. Проверка новых запросов

Посмотрите, как модель распределяет новые фразы по классам. Обратите внимание: вероятность модели — не обязательно калиброванная вероятность реального события.

In [ ]:
if "model" in globals():
    new_texts = [
        "оценить вероятность дефолта по кредитной истории",
        "помочь клиенту восстановить доступ к приложению",
        "снизить просрочку по кредитному портфелю"
    ]
    print(list(zip(new_texts, model.predict(new_texts))))
else:
    print("Сначала выполните ячейку обучения классификатора.")

## 5. Эмбеддинги: что меняется по сравнению с TF–IDF?

TF–IDF представляет текст через веса слов и n-грамм. Эмбеддинг — плотный вектор, полученный обученной моделью; он может кодировать контекстуальные и семантические закономерности. Для получения качественных эмбеддингов обычно используют предварительно обученную модель, например Sentence Transformers или модели семейства BERT. Здесь внешний доступ не требуется: мы фиксируем концепцию и формулируем критерии выбора модели.

При выборе модели проверяйте: язык и домен текстов, лицензию, размер, скорость, способ агрегирования предложений и качество на вашей задаче. Не сравнивайте векторы разных моделей без специального обоснования.

### Задание на бумаге: TF–IDF или эмбеддинги?
1. «Клиент не вернул кредит» и «Заёмщик допустил дефолт» могут иметь мало общих слов. Какой подход потенциально лучше уловит близость смысла?
2. Почему даже эмбеддинг-модель может ошибаться на финансовых терминах?
3. Какие 20 примеров вы собрали бы для проверки качества поиска похожих текстов?

## 6. Практическое задание
1. Добавьте не менее 10 собственных коротких текстов и укажите метки классов.
2. Сравните результаты модели с разными `ngram_range` (например, `(1, 1)` и `(1, 2)`).
3. Найдите ошибочно классифицированные примеры.
4. Обсудите риск утечки данных и дисбаланса классов.
5. Сформулируйте, по каким метрикам оценивать задачу: precision, recall, F1 и матрица ошибок.

**Важно:** демонстрационный корпус слишком мал для выводов о качестве модели в реальном применении.